# BnSentMix — Model 1: Word + Character TF-IDF Linear SVM

**Research question:** Under a duplicate-group-disjoint protocol, can explicitly modelling positive and negative evidence improve detection of the `mixed` class over strong flat classifiers?

**Role of Model 1:** establish a fast, resource-efficient lexical baseline. Later models must be compared on the same locked split using **macro-F1 (primary)** and **mixed-class F1 (key secondary)**.

The label mapping was already confirmed: `0=positive`, `1=negative`, `2=neutral`, `3=mixed`. This notebook does not repeat label inspection or correction.

## Fresh six-experiment plan

| Model | Experiment | Hypothesis tested |
|---|---|---|
| M1 | Word + character TF-IDF Linear SVM | Surface lexical and spelling patterns form a strong non-neural baseline. |
| M2 | Flat 4-class XLM-R | Contextual pretraining improves over sparse surface features. |
| M3 | XLM-R with class-balanced/focal loss | Any mixed-class gain may be explained by imbalance handling alone. |
| M4 | Dual-polarity XLM-R | Explicit positive/negative evidence heads improve mixed sentiment beyond imbalance correction. |
| M5 | M4 + clause/contrast-aware pooling | Localizing opposite evidence across clauses further improves mixed detection. |
| M6 | M5 + romanization-noise consistency | Consistency training improves robustness without sacrificing clean performance. |

Do not inspect the final-test scores until all six model definitions and hyperparameters are frozen.

In [ ]:
from pathlib import Path
import hashlib, json, re, unicodedata

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, f1_score
)

# Declared before training
DATA_PATH = Path('/kaggle/input/datasets/olidopu/bnsentmix/dataset.csv')
PROTOCOL_DIR = Path('/kaggle/working/bnsentmix_protocol_2026')
MODEL_DIR = Path('/kaggle/working/bnsentmix_model1')
PROTOCOL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

SEED = 2026
LABELS = [0, 1, 2, 3]
LABEL_NAMES = {0: 'positive', 1: 'negative', 2: 'neutral', 3: 'mixed'}
MIXED_LABEL = 3
C_VALUES = [0.5, 1.0, 2.0]
OPEN_FINAL_TEST = False  # Keep False until M1-M6 are frozen.

print('Configuration locked. Final test open:', OPEN_FINAL_TEST)

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def normalized_group_key(text):
    text = unicodedata.normalize('NFKC', str(text)).lower()
    text = re.sub(r'[\u200b-\u200d\ufeff]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return hashlib.sha256(text.encode('utf-8')).hexdigest()

raw = pd.read_csv(DATA_PATH)
lookup = {column.strip().lower(): column for column in raw.columns}
text_col = next(lookup[name] for name in ['sentence', 'text', 'comment'] if name in lookup)
label_col = next(lookup[name] for name in ['label', 'sentiment', 'class'] if name in lookup)

df = raw[[text_col, label_col]].rename(columns={text_col: 'text', label_col: 'label'}).copy()
df.insert(0, 'source_row_id', np.arange(len(df)))
df = df.dropna(subset=['text', 'label'])
df['text'] = df['text'].astype(str).str.strip()
df['label'] = df['label'].astype(int)
df = df[df['text'].ne('')].copy()
df['group_id'] = df['text'].map(normalized_group_key)

# Remove only repeated copies of the same normalized text-label pair.
# If identical text has different confirmed labels, keep both rows in one split group.
rows_before = len(df)
df = df.drop_duplicates(['group_id', 'label'], keep='first').reset_index(drop=True)

RAW_SHA256 = sha256_file(DATA_PATH)
print({'raw_rows': len(raw), 'modeling_rows': len(df),
       'same_label_duplicates_removed': rows_before - len(df),
       'raw_sha256': RAW_SHA256})

In [ ]:
# 20 group-stratified folds -> 14/3/3 folds = approximately 70/15/15.
splitter = StratifiedGroupKFold(n_splits=20, shuffle=True, random_state=SEED)
fold_ids = np.full(len(df), -1, dtype=int)
for fold_id, (_, held_out) in enumerate(
    splitter.split(df['text'], df['label'], groups=df['group_id'])
):
    fold_ids[held_out] = fold_id

df['split'] = np.where(
    fold_ids < 14, 'train', np.where(fold_ids < 17, 'validation', 'test')
)
assert df.groupby('group_id')['split'].nunique().max() == 1

split_path = PROTOCOL_DIR / 'bnsentmix_locked_split.csv'
df.to_csv(split_path, index=False)
SPLIT_SHA256 = sha256_file(split_path)
manifest = {
    'raw_sha256': RAW_SHA256, 'split_sha256': SPLIT_SHA256, 'seed': SEED,
    'strategy': '20-fold StratifiedGroupKFold; 14/3/3 folds',
    'group_key': 'SHA256(NFKC + lowercase + collapsed whitespace)',
    'label_names': LABEL_NAMES,
    'split_counts': df['split'].value_counts().to_dict(),
}
with open(PROTOCOL_DIR / 'split_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(manifest, handle, indent=2)

display(pd.crosstab(df['split'], df['label']).rename(columns=LABEL_NAMES))
print('Split SHA-256:', SPLIT_SHA256)

In [ ]:
train_df = df[df['split'].eq('train')].copy()
val_df = df[df['split'].eq('validation')].copy()
test_df = df[df['split'].eq('test')].copy()

features = FeatureUnion([
    ('word', TfidfVectorizer(
        lowercase=True, ngram_range=(1, 2), min_df=2, max_df=0.98,
        sublinear_tf=True, max_features=60_000
    )),
    ('char', TfidfVectorizer(
        analyzer='char_wb', lowercase=True, ngram_range=(3, 5), min_df=2,
        sublinear_tf=True, max_features=120_000
    )),
])
X_train = features.fit_transform(train_df['text'])
X_val = features.transform(val_df['text'])
y_train = train_df['label'].to_numpy()
y_val = val_df['label'].to_numpy()
print('Train features:', X_train.shape, '| Validation features:', X_val.shape)

In [ ]:
models, rows = {}, []
for c_value in C_VALUES:
    model = LinearSVC(C=c_value, class_weight=None, max_iter=10_000, random_state=SEED)
    model.fit(X_train, y_train)
    prediction = model.predict(X_val)
    rows.append({
        'C': c_value,
        'macro_f1': f1_score(y_val, prediction, labels=LABELS, average='macro'),
        'mixed_f1': f1_score(y_val == MIXED_LABEL, prediction == MIXED_LABEL),
        'accuracy': accuracy_score(y_val, prediction),
    })
    models[c_value] = model

validation_results = pd.DataFrame(rows).sort_values(
    ['macro_f1', 'mixed_f1', 'C'], ascending=[False, False, True]
).reset_index(drop=True)
best_c = float(validation_results.loc[0, 'C'])
best_model = models[best_c]
val_prediction = best_model.predict(X_val)
display(validation_results.round(4))
print('Selected C:', best_c)

In [ ]:
report = classification_report(
    y_val, val_prediction, labels=LABELS,
    target_names=[LABEL_NAMES[label] for label in LABELS],
    output_dict=True, zero_division=0
)
display(pd.DataFrame(report).T.round(4))
display(pd.DataFrame(
    confusion_matrix(y_val, val_prediction, labels=LABELS),
    index=[f'true_{LABEL_NAMES[x]}' for x in LABELS],
    columns=[f'pred_{LABEL_NAMES[x]}' for x in LABELS],
))

validation_results.to_csv(MODEL_DIR / 'validation_search.csv', index=False)
val_output = val_df[['source_row_id', 'group_id', 'text', 'label']].copy()
val_output['prediction'] = val_prediction
val_output.to_csv(MODEL_DIR / 'validation_predictions.csv', index=False)
joblib.dump({
    'features': features, 'classifier': best_model, 'selected_C': best_c,
    'label_names': LABEL_NAMES, 'raw_sha256': RAW_SHA256,
    'split_sha256': SPLIT_SHA256,
}, MODEL_DIR / 'model1_tfidf_linear_svm.joblib', compress=3)
print('Validation artifacts saved to:', MODEL_DIR)

## Final-test gate

Leave `OPEN_FINAL_TEST=False` during development. After M1–M6 configurations are frozen, change it once and run the next cell for the final comparison. Do not tune anything after seeing these scores.

In [ ]:
if not OPEN_FINAL_TEST:
    print('Final test remains sealed.')
else:
    X_test = features.transform(test_df['text'])
    y_test = test_df['label'].to_numpy()
    test_prediction = best_model.predict(X_test)
    test_metrics = {
        'accuracy': accuracy_score(y_test, test_prediction),
        'macro_f1': f1_score(y_test, test_prediction, labels=LABELS, average='macro'),
        'weighted_f1': f1_score(y_test, test_prediction, labels=LABELS, average='weighted'),
        'mixed_f1': f1_score(y_test == MIXED_LABEL, test_prediction == MIXED_LABEL),
    }
    print(test_metrics)
    test_output = test_df[['source_row_id', 'group_id', 'text', 'label']].copy()
    test_output['prediction'] = test_prediction
    test_output.to_csv(MODEL_DIR / 'FINAL_test_predictions.csv', index=False)
    with open(MODEL_DIR / 'FINAL_test_metrics.json', 'w') as handle:
        json.dump(test_metrics, handle, indent=2)